# Day 006：Attention、Q/K/V 与单 Head

这是 Day 006 的可运行实验手册，与 [Day 006 互动档案](../day-006-2026-08-20.md) 配套使用。

运行方式：按顺序执行 Cell；每个实验只验证一个小问题。不要把看到代码运行成功当成已经理解，先阅读 Cell 上方的说明，再观察输出并用自己的话解释。

本 Notebook 覆盖：加权读取、Softmax、Q/K/V、缩放点积、Causal Mask、整句 shape 和单 Head 输出。暂不进入 GQA、RoPE、KV Cache 或 MiniMind 源码。

## 1. 先人工完成一次加权读取

当前正在更新“苹果”。四个位置的信息向量和人工读取比例如下。这里先不问模型如何得到比例，只观察比例怎样合并信息。

In [1]:
import torch

values = torch.tensor([
    [1.0, 0.0],  # 我
    [0.0, 2.0],  # 吃了
    [1.0, 1.0],  # 一个
    [2.0, 0.0],  # 苹果
])
weights = torch.tensor([0.1, 0.6, 0.1, 0.2])
read_result = weights @ values
print("加权读取结果：", read_result.tolist())

加权读取结果： [0.6, 1.3]


`weights` 回答“读多少”，`values` 回答“读什么”。一个读取者把多个来源的向量按比例相加，得到一份属于自己的综合信息。

## 2. Softmax：匹配分数变成读取比例

Softmax 的职责只有一个：接收匹配分数，输出非负且总和为 1 的读取比例。下面对照手工的 `exp(score) / exp(score).sum()` 与 PyTorch 实现。

In [2]:
scores = torch.tensor([1.0, 3.0, 1.0, 2.0])
positive_scores = torch.exp(scores)
manual_weights = positive_scores / positive_scores.sum()
torch_weights = torch.softmax(scores, dim=0)
print("匹配分数：", scores)
print("e^分数：", positive_scores)
print("手工 Softmax：", manual_weights)
print("PyTorch Softmax：", torch_weights)
print("比例总和：", torch_weights.sum())
print("两种算法是否一致：", torch.allclose(manual_weights, torch_weights))

匹配分数： tensor([1., 3., 1., 2.])
e^分数： tensor([ 2.7183, 20.0855,  2.7183,  7.3891])
手工 Softmax： tensor([0.0826, 0.6103, 0.0826, 0.2245])
PyTorch Softmax： tensor([0.0826, 0.6103, 0.0826, 0.2245])
比例总和： tensor(1.0000)
两种算法是否一致： True


## 3. 同一个完整向量分别产生 Q、K、V

Embedding 先根据 token ID 查到完整向量 `h`。Q/K/V 不是把 Embedding 切成三段，而是把同一个 `h` 分别送入三套 Linear。

In [3]:
h = torch.tensor([1.0, 2.0, 3.0])
Wq = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
Wk = torch.tensor([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
Wv = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
Q = h @ Wq.T
K = h @ Wk.T
V = h @ Wv.T
print("Q：", Q)
print("K：", K)
print("V：", V)

Q： tensor([1., 2.])
K： tensor([2., 3.])
V： tensor([4., 5.])


Q 表示当前读取者想找什么，K 表示某个位置适合被什么需求匹配，V 表示该位置真正提供的内容。Q/K 点积只产生匹配分数，V 直到比例确定后才参与加权读取。

In [4]:
current_q = torch.tensor([1.0, 2.0])
keys = torch.tensor([[2.0, 3.0], [4.0, 1.0]])
values = torch.tensor([[10.0, 0.0], [0.0, 10.0]])
scores = current_q @ keys.T
weights = torch.softmax(scores, dim=0)
result = weights @ values
print("匹配分数：", scores)
print("读取比例：", weights)
print("读取结果：", result)

匹配分数： tensor([8., 6.])
读取比例： tensor([0.8808, 0.1192])
读取结果： tensor([8.8080, 1.1920])


这里的 `[8.8080, 1.1920]` 是当前“苹果”从上下文汇总读到的信息，不是模型参数，也不是完整 Transformer 后的最终 token 表示。

## 4. Causal Mask：未来位置不能参与读取

Decoder-only 模型让每个位置只能看自己和过去。遮罩必须在 Softmax 前执行：未来分数设为 `-inf`，这样 `exp(-inf)=0`。

In [5]:
scores = torch.tensor([2.0, 1.0, 3.0, 9.0])
without_mask = torch.softmax(scores, dim=0)
masked_scores = scores.clone()
masked_scores[3] = float("-inf")
with_mask = torch.softmax(masked_scores, dim=0)
print("无遮罩：", without_mask)
print("遮罩后：", with_mask)
print("遮罩后总和：", with_mask.sum())

无遮罩： tensor([0.0009, 0.0003, 0.0025, 0.9963])
遮罩后： tensor([0.2447, 0.0900, 0.6652, 0.0000])
遮罩后总和： tensor(1.)


## 5. 为什么要除以 `sqrt(head_dim)`

Q/K 点积是多个维度乘积的和。维度越大，分数的典型波动约按 `sqrt(d)` 增长；除以 `sqrt(d)` 可以让进入 Softmax 的分数保持相近尺度。

In [6]:
torch.manual_seed(0)
sample_count = 10000
for dimension in [2, 8, 32, 128]:
    q = torch.randn(sample_count, dimension)
    k = torch.randn(sample_count, dimension)
    raw_scores = (q * k).sum(dim=1)
    scaled_scores = raw_scores / (dimension ** 0.5)
    print(
        f"维度={dimension} "
        f"原始标准差约={raw_scores.std().item():.2f} "
        f"缩放后约={scaled_scores.std().item():.2f}"
    )

维度=2 原始标准差约=1.43 缩放后约=1.01
维度=8 原始标准差约=2.80 缩放后约=0.99
维度=32 原始标准差约=5.70 缩放后约=1.01
维度=128 原始标准差约=11.30 缩放后约=1.00


## 6. 四 token 的完整单 Head Attention

这段代码把前面的步骤连接起来：`h -> Q/K/V -> 缩放点积 -> Causal Mask -> Softmax -> weights @ V`。本实验特意让 `V` 为 2 维，因此单 Head 的中间结果也是 2 维；它还不是完整 Attention 子层的最终 `hidden_size` 输出。

In [7]:
torch.set_printoptions(precision=4, sci_mode=False)
h = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
    [1.0, 2.0, 1.0],
])
Wq = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
Wk = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
Wv = torch.tensor([[1.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
Q = h @ Wq.T
K = h @ Wk.T
V = h @ Wv.T
head_dim = Q.shape[-1]
scores = (Q @ K.T) / (head_dim ** 0.5)
future_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(future_mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
result = weights @ V
print("Q shape：", Q.shape)
print("K shape：", K.shape)
print("V shape：", V.shape)
print("scores shape：", scores.shape)
print("result shape：", result.shape)
print("每一行比例之和：", weights.sum(dim=-1))

Q shape： torch.Size([4, 2])
K shape： torch.Size([4, 2])
V shape： torch.Size([4, 2])
scores shape： torch.Size([4, 4])
result shape： torch.Size([4, 2])
每一行比例之和： tensor([1., 1., 1., 1.])


本 Notebook 的核心 shape 链：

```text
h              [sequence, hidden_in]
Q/K/V          [sequence, head_dim/value_dim]
Q @ K.T        [sequence, sequence]
weights @ V    [sequence, value_dim]
```

下一步再把单个 Head 扩展为多个 Head，并解释它们怎样拼接、怎样通过 `o_proj` 映射回模型的 `hidden_size`。